# Sparse linear regression — N=2000, D=200, K=5

In [ ]:
import os, json
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from scipy.special import logsumexp
from scipy.stats import norm as sp_norm
from sklearn.metrics import f1_score, precision_score, recall_score, roc_curve, auc

%matplotlib inline

# ── paths ──────────────────────────────────────────────────────────────────
RESULTS_DIR = Path("results/sparse_regression/N2000_D200_K5_Gauss_seed0")

STYLE = {
    "NUTS":        ("C2", "D",  False),
    "NUTS-HS":     ("C4", "P",  False),
    "Boom":        ("C0", "o",  False),
    "Sticky-Boom": ("C1", "s",  True),
    "ZZ":          ("C9", "v",  False),
    "Sticky-ZZ":   ("C3", "^",  True),
}

# ── load .pt files ──────────────────────────────────────────────────────────
runs = {}
for pt in sorted(RESULTS_DIR.glob("*.pt")):
    name = pt.stem
    payload = torch.load(pt, weights_only=False)
    color, marker, sticky = STYLE.get(name, ("C5", "o", False))
    runs[name] = {
        "name":    name,
        "samples": payload["samples"].cpu().numpy(),  # (S, 201)
        "wall":    float(payload["wall"]),
        "config":  payload["config"],
        "color":   color,
        "marker":  marker,
        "sticky":  sticky,
    }

# ── load metrics.json ───────────────────────────────────────────────────────
with open(RESULTS_DIR / "metrics.json") as f:
    meta = json.load(f)

cfg        = meta["config"]
true_coefs = np.array(meta["true_coefs"])   # shape (201,): [intercept, beta_1..beta_200]
is_signal  = np.array(meta["is_signal"])    # shape (201,): 5 True entries

# convenience indices
sig_idx = np.where(is_signal)[0]      # columns in samples that are true signals
nul_idx = np.where(~is_signal)[0]     # null columns (includes intercept col 0)

print(f"Loaded samplers: {list(runs)}")
print(f"N={cfg['N']}, D={cfg['D']}, K={cfg['n_signals']}, seed={cfg['seed']}")
print(f"Signal indices: {sig_idx}")
print(f"True signal values: {true_coefs[sig_idx].round(3)}")


## Introduction

High-dimensional sparse linear regression: N=2000 observations, D=200 predictors, K=5 true signals (the rest exactly zero), Gaussian prior. We compare six samplers from two families:

- **Non-sticky** (Boom, ZZ, NUTS): target the standard Gaussian posterior π(β|y). No coordinate is exactly zero in this posterior; variable selection requires a thresholding rule on the marginals.
- **Sticky** (Sticky-Boom, Sticky-ZZ) and **NUTS-HS**: intrinsically sparse. Sticky samplers target a mixed measure with atomic mass at zero; NUTS-HS uses a horseshoe prior that places very heavy density near zero.

Key question: which family does better variable selection in D=200?


In [ ]:
## Build test set (same DGP seed as the experiment, seed+1 for test)
N_te = cfg["N"]
D    = cfg["D"]          # 200
lik_noise_std = cfg["lik_noise_std"]

rng = np.random.default_rng(cfg["seed"] + 1)
X_te_raw = rng.normal(size=(N_te, D))
X_te_raw = (X_te_raw - X_te_raw.mean(0)) / X_te_raw.std(0)

# true model: intercept=0, betas = true_coefs[1:]
beta_true = true_coefs[1:]   # shape (200,)
int_true  = true_coefs[0]    # 0.0

y_clean = X_te_raw @ beta_true + int_true
y_noisy = y_clean + lik_noise_std * rng.normal(size=N_te)

# Augment X with intercept column for dot with full samples[:,0..200]
X_te = np.column_stack([np.ones(N_te), X_te_raw])  # (N_te, 201)

print(f"Test set: X_te {X_te.shape}, y_clean std={y_clean.std():.3f}")


## Bayesian metrics table


In [ ]:
def elpd_for(samples):
    """
    ELPD = mean_i log( (1/S) sum_s N(y_i | x_i @ beta_s, sigma^2) )
    Uses logsumexp for numerical stability.
    samples : (S, 201), X_te : (N, 201)
    """
    S = samples.shape[0]
    # preds[s, i] = x_i @ beta_s
    preds = X_te @ samples.T          # (N, S)
    # log N(y | mu, sigma^2) = -0.5*log(2pi*sigma^2) - (y-mu)^2/(2*sigma^2)
    log2pisig2 = np.log(2 * np.pi * lik_noise_std**2)
    log_liks = -0.5 * log2pisig2 - 0.5 * ((y_noisy[:, None] - preds) ** 2) / lik_noise_std**2
    # log( mean_s exp(log_lik_s) ) = logsumexp - log(S)
    return float(np.mean(logsumexp(log_liks, axis=1) - np.log(S)))


def p_zero_vec(samples):
    """Per-coordinate fraction of exactly-zero draws."""
    return (np.abs(samples) < 1e-10).mean(0)


def f1_ci(samples, alpha=0.05):
    lo = np.quantile(samples, alpha / 2,     axis=0)
    hi = np.quantile(samples, 1 - alpha / 2, axis=0)
    selected = (lo > 0) | (hi < 0)
    return float(f1_score(is_signal, selected, zero_division=0))


def f1_p0(samples, thr=0.5):
    pz = p_zero_vec(samples)
    selected = pz <= thr
    return float(f1_score(is_signal, selected, zero_division=0))


rows = []
for name, r in runs.items():
    s   = r["samples"]
    mu  = s.mean(0)
    pz  = p_zero_vec(s)
    row = {
        "sampler":         name,
        "n_draws":         s.shape[0],
        "wall_sec":        round(r["wall"], 1),
        "post_mean_rmse":  float(np.sqrt(np.mean((mu - true_coefs)**2))),
        "elpd":            elpd_for(s),
        "pred_rmse":       float(np.sqrt(np.mean((X_te @ mu - y_clean)**2))),
        "p0_null":         float(pz[nul_idx].mean()) if r["sticky"] else float("nan"),
        "p0_signal":       float(pz[sig_idx].mean()) if r["sticky"] else float("nan"),
        "f1_ci":           f1_ci(s),
        "f1_p0":           f1_p0(s) if r["sticky"] else float("nan"),
    }
    rows.append(row)

metrics_df = (pd.DataFrame(rows)
              .set_index("sampler")
              .sort_values("elpd", ascending=False))

metrics_df.round(4)


## Two different posteriors

**Non-sticky samplers (Boom, ZZ) and NUTS** target the standard Gaussian posterior π(β|y) ∝ N(y|Xβ, σ²) · N(β|0, τ²I). In this posterior no coordinate is exactly zero — every null β_j has a posterior that is a narrow Gaussian centred near 0, not a point mass. Variable selection via a 95% CI therefore asks "is this narrow Gaussian far enough from zero?" and will often fail for weak signals in high D, giving F1 ≈ 0.55–0.67.

**Sticky samplers** target a *mixed measure* π̃ that adds atomic mass at zero: each coordinate can be exactly zero with some probability κ. This is not an approximation failure — it is a different probabilistic object designed to express sparsity naturally. The sticky posterior's marginals will show a spike at zero (the histogram looks bimodal with a delta at 0), and that is correct.

**NUTS-HS** achieves similar variable selection via a horseshoe prior, which places very heavy mass near zero without a literal point mass. It targets yet another posterior. All three posteriors give similar predictive performance (ELPD), but very different sparsity structure.


## Signal coefficient posteriors


In [ ]:
n_sig = len(sig_idx)
ncols = min(n_sig, 5)
nrows = int(np.ceil(n_sig / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(3.4 * ncols, 3.0 * nrows), squeeze=False)
axes_flat = axes.flatten()

ordered_names = ["Boom", "ZZ", "NUTS", "NUTS-HS", "Sticky-Boom", "Sticky-ZZ"]
ordered_runs  = [runs[n] for n in ordered_names if n in runs]
x_pos = np.arange(len(ordered_runs))

for panel_i, col in enumerate(sig_idx):
    ax = axes_flat[panel_i]
    truth = true_coefs[col]

    for j, r in enumerate(ordered_runs):
        samp_c = r["samples"][:, col]
        parts = ax.violinplot(samp_c, positions=[j], widths=0.75,
                              showextrema=False, showmedians=False)
        for body in parts["bodies"]:
            body.set_facecolor(r["color"])
            body.set_edgecolor(r["color"])
            body.set_alpha(0.55)
        ax.scatter([j], [np.median(samp_c)], color=r["color"],
                   marker=r["marker"], s=22, edgecolor="white",
                   linewidths=0.4, zorder=4)
        # Annotate P(=0) for sticky
        if r["sticky"]:
            pz = (np.abs(samp_c) < 1e-10).mean()
            ax.text(j, ax.get_ylim()[0] if panel_i == 0 else truth - abs(truth)*1.6,
                    f"P₀={pz:.2f}", ha="center", va="top", fontsize=6.5,
                    color=r["color"])

    ax.axhline(truth, color="red", lw=1.5, ls="--", alpha=0.9)
    ax.axhline(0, color="grey", lw=0.5, alpha=0.4)
    ax.set_xticks(x_pos)
    ax.set_xticklabels([r["name"] for r in ordered_runs],
                       rotation=40, ha="right", fontsize=7)
    ax.set_title(rf"$\beta_{{{col}}}$  (true={truth:.2f})", fontsize=10)
    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

for k in range(n_sig, len(axes_flat)):
    axes_flat[k].set_visible(False)

fig.suptitle("Signal coordinate posteriors — red dashed = true value", fontsize=11)
plt.tight_layout()
plt.show()


## Null shrinkage

How tightly do the 195 null coefficients get pushed to zero? Non-sticky methods produce narrow but non-zero posteriors; sticky methods send most nulls to exactly zero.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.6 * len(ordered_runs) + 1.5))

rng_jit = np.random.default_rng(42)
for i, r in enumerate(ordered_runs):
    null_means = r["samples"][:, nul_idx].mean(0)
    jitter = rng_jit.uniform(-0.18, 0.18, size=len(null_means))
    ax.scatter(null_means, np.full_like(null_means, i) + jitter,
               color=r["color"], marker=r["marker"],
               s=14, alpha=0.45, edgecolor="none")
    q25, q50, q75 = np.quantile(null_means, [0.25, 0.50, 0.75])
    ax.plot([q25, q75], [i, i], color="black", lw=3.0,
            solid_capstyle="butt", zorder=4)
    ax.scatter([q50], [i], color="black", marker="|", s=160, lw=2.5, zorder=5)

    # Summary stats
    frac_near_zero = float((np.abs(null_means) < 0.01).mean())
    label = f"  {frac_near_zero*100:.0f}% |μ|<0.01"
    if r["sticky"]:
        pz_null = float((np.abs(r["samples"][:, nul_idx]) < 1e-10).mean())
        label += f",  P(=0)={pz_null:.2f}"
    ax.text(ax.get_xlim()[1] if i == 0 else 0.12, i - 0.35,
            label, fontsize=7.5, va="top", color=r["color"])

ax.axvline(0, color="red", lw=1.0, alpha=0.7)
ax.set_yticks(np.arange(len(ordered_runs)))
ax.set_yticklabels([r["name"] for r in ordered_runs])
ax.set_xlabel("posterior mean on null coordinates")
ax.set_title(f"Null shrinkage — {len(nul_idx)} true-zero coordinates  (bar = IQR, tick = median)")
ax.invert_yaxis()
ax.grid(axis="x", alpha=0.25)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.show()


## Slab check: conditional posteriors on signal coordinates

For sticky samplers the marginal at a signal coordinate is a mixture: a point mass at 0 (the spike) plus a continuous slab. Conditioning on β ≠ 0 extracts the slab. We compare this slab to the NUTS posterior on the same coordinate — if the non-zero draws are consistent with the Gaussian posterior, the sticky sampler is behaving correctly as a *different* model but with a sensible slab.


In [ ]:
nuts_samples = runs["NUTS"]["samples"]  # reference

sticky_runs = {n: r for n, r in runs.items() if r["sticky"]}

n_sticky = len(sticky_runs)
fig, axes = plt.subplots(n_sticky, n_sig,
                         figsize=(3.2 * n_sig, 3.0 * n_sticky),
                         squeeze=False)

for row_i, (name, r) in enumerate(sticky_runs.items()):
    for col_i, sig_col in enumerate(sig_idx):
        ax = axes[row_i][col_i]

        # Slab: draws where beta != 0
        samp_col = r["samples"][:, sig_col]
        slab = samp_col[np.abs(samp_col) > 1e-10]

        if len(slab) > 5:
            ax.hist(slab, bins=40, density=True, color=r["color"],
                    alpha=0.55, label=f"slab (n={len(slab)})")

        # NUTS reference curve
        nuts_col = nuts_samples[:, sig_col]
        mu_n = nuts_col.mean()
        sd_n = nuts_col.std()
        xs = np.linspace(mu_n - 4 * sd_n, mu_n + 4 * sd_n, 300)
        ax.plot(xs, sp_norm.pdf(xs, mu_n, sd_n),
                color="C2", lw=1.8, ls="-", label="NUTS N(μ,σ²)")

        # True value
        ax.axvline(true_coefs[sig_col], color="red", lw=1.4, ls="--", alpha=0.9)

        pz = float((np.abs(samp_col) < 1e-10).mean())
        ax.set_title(rf"$\beta_{{{sig_col}}}$  P₀={pz:.2f}", fontsize=9)
        ax.tick_params(labelsize=7)
        ax.legend(fontsize=6.5, frameon=False)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

    axes[row_i][0].set_ylabel(name, fontsize=10)

fig.suptitle("Slab component vs NUTS posterior — signal coordinates\n"
             "(red dashed = true value, green = NUTS marginal)", fontsize=10)
plt.tight_layout()
plt.show()


## ESS table

Geyer IPS ESS across all 201 dimensions. For sticky samplers we additionally report:
- **active ESS**: ESS on the slab draws only (per-coordinate, non-zero samples)
- **indicator ESS**: ESS of the 0/1 inclusion indicator 1[β≠0]


In [ ]:
def ess_ips(samples):
    """Geyer IPS ESS, vectorised over dimensions. samples: (n, d)."""
    x = samples - samples.mean(0, keepdims=True)
    n, d = x.shape
    var = (x ** 2).mean(0)
    rho_sum   = np.zeros(d)
    prev_pair = np.full(d, np.inf)
    active    = np.ones(d, dtype=bool)
    k = 1
    while k + 1 <= min(n - 1, 500):
        ck   = (x[:n - k]     * x[k:]    ).mean(0) / np.maximum(var, 1e-30)
        ckp1 = (x[:n - k - 1] * x[k + 1:]).mean(0) / np.maximum(var, 1e-30)
        pair = ck + ckp1
        kill = active & ((pair <= 0) | (pair >= prev_pair))
        active    = active & ~kill
        rho_sum  += np.where(active, pair, 0.0)
        prev_pair = np.where(active, pair, prev_pair)
        if not active.any():
            break
        k += 2
    return n / np.clip(1.0 + 2.0 * rho_sum, 1.0, None)


ess_rows = []
for name, r in runs.items():
    s    = r["samples"]
    wall = r["wall"]
    ess  = ess_ips(s)

    row = {
        "sampler":      name,
        "n_draws":      s.shape[0],
        "wall_sec":     round(wall, 1),
        "ESS_min":      round(ess.min(), 1),
        "ESS_median":   round(np.median(ess), 1),
        "ESS/sec_min":  round(ess.min() / wall, 2),
        "ESS/sec_med":  round(np.median(ess) / wall, 2),
    }

    if r["sticky"]:
        # indicator ESS: 1[beta != 0]
        ind = (np.abs(s) > 1e-10).astype(float)
        ess_ind = ess_ips(ind)
        row["ind_ESS_min"]    = round(ess_ind.min(), 1)
        row["ind_ESS_median"] = round(np.median(ess_ind), 1)
    else:
        row["ind_ESS_min"]    = float("nan")
        row["ind_ESS_median"] = float("nan")

    ess_rows.append(row)

ess_df = pd.DataFrame(ess_rows).set_index("sampler")
ess_df
